![IITIS](pictures/logoIITISduze.png)

# Wyżarzanie równoległe

In [96]:
# Implementation of the algorithm
import random
import numpy as np
from typing import Optional

def project(x: float):
    if x == 0:
        return random.choice([-1, 1])
    else:
        return np.sign(x)


projector = np.vectorize(project)
clip = np.vectorize(lambda x: max(-1.0, min(1.0,x)))


def calculate_gradient(J: np.ndarray, h: np.ndarray, x: np.ndarray, state: np.ndarray, lambda_t: float) -> np.ndarray:
    return J @ state + h + lambda_t * x


def calculate_momentum(momentum, momentum_constant, step_size, gradient):
    return 


def calculate_energy(J: np.ndarray, h: np.ndarray, state: np.ndarray):
    return state @ J @ state.T + state @ h 


def parrarel_annealing(J, h, step_size: float, lambda_t_max: float, num_steps: int, schedule: Optional[np.ndarray] = None):
    n = len(h)
    x = np.zeros(n)
    momentum = 0

    if schedule is None:
        schedule = np.geomspace(lambda_t_max, 1e-12, num=num_steps)

    for k in range(num_steps):
        x = clip(x)
        lambda_t = schedule[k]
        state = projector(x)
        gradient = calculate_gradient(J, h, x, state, lambda_t)
        momentum = momentum * 0.99 - step_size * gradient
        momentum = clip(momentum)
        x += momentum

    return projector(x), calculate_energy(J, h, projector(x))



    


In [112]:
n = 10

scaling_func = np.vectorize(lambda x: 2*x-1)  # przesunięcie wartości z (0, 1) do (-1, 1)
J = np.triu(scaling_func(np.random.rand(n, n)))  # losowa gęsta macierz górnotrójkątna
h = scaling_func(np.random.rand(n))  # losowy wektor




In [116]:
from dimod import BinaryQuadraticModel
from dwave.samplers import SimulatedAnnealingSampler

bqm_instance = BinaryQuadraticModel(h, J, vartype="SPIN")
sampler= SimulatedAnnealingSampler()
sampleset = sampler.sample(bqm_instance, num_reads=1)
print(sampleset)
print(parrarel_annealing(J, h, 10, 0.005, 10000))

   0  1  2  3  4  5  6  7  8  9    energy num_oc.
0 -1 +1 -1 -1 +1 +1 -1 -1 -1 +1 -9.604505       1
['SPIN', 1 rows, 1 samples, 10 variables]
(array([-1.,  1., -1., -1.,  1.,  1., -1., -1.,  1.,  1.]), np.float64(-9.604505323873457))
